# LlamaIndex Agentic RAG (SEC v2 Parity)

This notebook ports the LangGraph SEC v2 state machine to a LlamaIndex event-driven Workflow.

Parity goals:
- Same retrieval stack: BM25 + Dense (Chroma) + RRF + CrossEncoder rerank.
- Same control flow: planner -> retriever -> context evaluator -> generator -> critic -> repair/retry.
- Same model strings and temperatures as the LangGraph notebook.
- No re-embedding: connect to existing Chroma collection via ChromaVectorStore.

In [ ]:
# Install/repair dependencies in the active notebook kernel if needed.
# %pip install -U chromadb python-dotenv rank-bm25 sentence-transformers pydantic
# %pip install -U llama-index-core llama-index-vector-stores-chroma
# %pip install -U llama-index-llms-groq llama-index-llms-gemini

  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---------- ----------------------------- 5.8/21.9 MB 28.2 MB/s eta 0:00:01
   --------------------- ------------------ 11.8/21.9 MB 28.6 MB/s eta 0:00:01
   -------------------------------- ------- 17.8/21.9 MB 28.7 MB/s eta 0:00:01
   ---------------------------------------  21.8/21.9 MB 29.1 MB/s eta 0:00:01
   ---------------------------------------- 21.9/21.9 MB 24.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 23.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 22.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ------------------ --------------------- 5.8/12.6 MB 28.8 MB/s eta 0:00:01
   -------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.2.14 requires openai<2.0.0,>=1.58.1, but you have openai 2.28.0 which is incompatible.


In [1]:
import os
import re
import json
import time
import threading
from dataclasses import dataclass
from pathlib import Path
from collections import deque
from typing import Any, Dict, List, Optional, TypedDict, Literal

import numpy as np
import pandas as pd
import chromadb

from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from pydantic import BaseModel, Field, field_validator, model_validator

from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.workflow import Workflow, Event, StartEvent, StopEvent, step, Context

from llama_index.llms.groq import Groq as LIGroq
from llama_index.llms.gemini import Gemini as LIGemini


# Resolve repo root robustly from notebook location.
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name.lower() == "langgraph" else CWD

# Load .env from repository root.
load_dotenv(PROJECT_ROOT / ".env")

CONFIG: Dict[str, Any] = {
    "random_seed": 42,

    # Provider switch and profile parity with LangGraph SEC v2.
    "provider": os.getenv("LLM_PROVIDER", "groq"),
    "execution_profile": os.getenv("EXECUTION_PROFILE", "final"),

    # Relative paths are read from .env and resolved against PROJECT_ROOT.
    "sec_chunks_path": os.getenv("SEC_CHUNKS_PATH", "sec_rag_team_share/sec_data/chunks/sec_chunks.jsonl"),
    "chroma_db_path": os.getenv("CHROMA_DB_PATH", "sec_rag_team_share/chroma_db"),
    "chroma_collection_name": os.getenv("CHROMA_COLLECTION_NAME", ""),

    # Retrieval and loop controls.
    "max_context_retries": 1,
    "max_repair_retrievals": 1,
    "bm25_top_k": 8,
    "dense_top_k": 8,
    "rerank_top_k": 5,
    "decomposition_top_k_per_subquery": 4,

    # Embedding/rerank model parity.
    "dense_model_name": "sentence-transformers/all-mpnet-base-v2",
    "reranker_model_name": "cross-encoder/ms-marco-MiniLM-L-6-v2",

    # Exact model strings from LangGraph SEC v2.
    "groq_dev_generator_model": "llama-3.1-8b-instant",
    "groq_dev_agent_model": "llama-3.1-8b-instant",
    "groq_dev_judge_model": "llama-3.1-8b-instant",
    "groq_final_generator_model": "meta-llama/llama-4-scout-17b-16e-instruct",
    "groq_final_agent_model": "llama-3.1-8b-instant",
    "groq_final_judge_model": "llama-3.1-8b-instant",

    "gemini_dev_generator_model": "gemini-2.0-flash-lite",
    "gemini_dev_agent_model": "gemini-2.0-flash-lite",
    "gemini_dev_judge_model": "gemini-2.0-flash-lite",
    "gemini_final_generator_model": "gemini-2.0-flash",
    "gemini_final_agent_model": "gemini-2.0-flash",
    "gemini_final_judge_model": "gemini-2.0-flash",

    # Temperature parity.
    "temperature_planner": 0.0,
    "temperature_agent": 0.0,
    "temperature_rewriter": 0.2,
    "temperature_context_eval": 0.0,
    "temperature_generator": 0.2,
    "temperature_critic": 0.0,
    "temperature_repair": 0.1,
    "temperature_judge": 0.0,

    # Context limits.
    "generator_max_context_chunks": 8,
    "generator_max_context_chars": 12000,
    "control_max_context_chunks": 4,
    "control_max_context_chars": 6000,

    # Misc.
    "enable_retrieval_sanity_check": True,
    "max_rpm": 28,
    "llm_max_retries": 3,
    "llm_retry_base_delay_sec": 5,
}

np.random.seed(CONFIG["random_seed"])


def resolve_path_from_env(path_like: str) -> Path:
    p = Path(path_like)
    return p if p.is_absolute() else (PROJECT_ROOT / p).resolve()


def resolve_model_name(role: str) -> str:
    provider = CONFIG["provider"]
    profile = CONFIG["execution_profile"]
    return CONFIG[f"{provider}_{profile}_{role}_model"]


def resolve_role_temperature(role: str, task_name: Optional[str] = None) -> float:
    if task_name:
        task_key = f"temperature_{task_name}"
        if task_key in CONFIG:
            return float(CONFIG[task_key])
    role_key = f"temperature_{role}"
    return float(CONFIG[role_key]) if role_key in CONFIG else 0.0


class RateLimiter:
    def __init__(self, max_rpm: int):
        self.max_rpm = max_rpm
        self.window = 60.0
        self._timestamps: deque = deque()
        self._lock = threading.Lock()

    def wait(self) -> None:
        with self._lock:
            now = time.time()
            while self._timestamps and now - self._timestamps[0] >= self.window:
                self._timestamps.popleft()
            if len(self._timestamps) >= self.max_rpm:
                sleep_for = self.window - (now - self._timestamps[0])
                if sleep_for > 0:
                    time.sleep(sleep_for)
            self._timestamps.append(time.time())


rate_limiter = RateLimiter(CONFIG["max_rpm"])


def make_llm(role: str, task_name: Optional[str] = None):
    model_name = resolve_model_name(role)
    temp = resolve_role_temperature(role, task_name)

    if CONFIG["provider"] == "groq":
        api_key = os.getenv("GROQ_API_KEY", "")
        if not api_key:
            raise ValueError("GROQ_API_KEY is missing in .env")
        return LIGroq(model=model_name, api_key=api_key, temperature=temp)

    if CONFIG["provider"] == "gemini":
        api_key = os.getenv("GOOGLE_API_KEY", "")
        if not api_key:
            raise ValueError("GOOGLE_API_KEY is missing in .env")
        return LIGemini(model=model_name, api_key=api_key, temperature=temp)

    raise ValueError(f"Unsupported provider: {CONFIG['provider']}")


planner_llm = make_llm("agent", task_name="planner")
context_eval_llm = make_llm("agent", task_name="context_eval")
generator_llm = make_llm("generator", task_name="generator")
critic_llm = make_llm("agent", task_name="critic")
repair_llm = make_llm("agent", task_name="repair")
judge_llm = make_llm("judge", task_name="judge")

print(f"Provider: {CONFIG['provider']}")
print(f"Execution profile: {CONFIG['execution_profile']}")
print(f"Generator model: {resolve_model_name('generator')}")
print(f"Agent model: {resolve_model_name('agent')}")
print(f"Planner temperature: {resolve_role_temperature('agent', 'planner')}")
print(f"Repair temperature: {resolve_role_temperature('agent', 'repair')}")

W0314 10:29:32.899000 17180 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Provider: groq
Execution profile: final
Generator model: meta-llama/llama-4-scout-17b-16e-instruct
Agent model: llama-3.1-8b-instant
Planner temperature: 0.0
Repair temperature: 0.1


c:\Users\wenxu\miniconda3\Lib\site-packages\llama_index\llms\gemini\base.py:21: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
@dataclass
class RetrievedChunk:
    doc_name: str
    company: str
    page_num: int
    chunk_id: str
    raw_chunk: str
    contextual_chunk: str
    score: float
    source: str
    ticker: Optional[str] = None


def load_sec_chunks(path: Path) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    if "flag_low_value_combined" in df.columns:
        df = df[~df["flag_low_value_combined"].fillna(False)].reset_index(drop=True)
    return df


def sec_df_to_chunk_dicts(df: pd.DataFrame) -> List[Dict[str, Any]]:
    chunks: List[Dict[str, Any]] = []
    for i, row in df.iterrows():
        doc_name = f"{row['ticker']}_{row['form_type']}_{str(row['filing_date'])[:10]}"
        contextual_text = (
            f"Company: {row['company_name']} ({row['ticker']})\n"
            f"Filing: {row['form_type']} | Date: {str(row['filing_date'])[:10]} | Year: {row['filing_year']}\n"
            f"Section: {row['section_title']}\n"
            f"Content: {row['text']}"
        )
        chunks.append(
            {
                "row_idx": i,
                "chunk_id_str": str(row["chunk_id"]),
                "ticker": row["ticker"],
                "company": row["company_name"],
                "doc_name": doc_name,
                "filing_year": int(row["filing_year"]),
                "form_type": row["form_type"],
                "page_num": int(row.get("chunk_index", 0)),
                "raw_chunk": row["text"],
                "contextual_chunk": contextual_text,
            }
        )
    return chunks


def _rrf_score(rank: int, k: int = 60) -> float:
    return 1.0 / (k + rank + 1)


def safe_int(value: Any, default: int = 0) -> int:
    try:
        return int(str(value))
    except (TypeError, ValueError):
        return default


class CorpusIndex:
    def __init__(self, chunks: List[Dict[str, Any]], chroma_db_path: Path, collection_name: str = ""):
        self.df = pd.DataFrame(chunks).copy()

        # BM25 side for sparse retrieval.
        self.df["bm25_tokens"] = self.df["contextual_chunk"].str.lower().str.split()
        self.bm25 = BM25Okapi(self.df["bm25_tokens"].tolist())

        # Dense side: existing ChromaDB collection via LlamaIndex ChromaVectorStore.
        self.chroma_client = chromadb.PersistentClient(path=str(chroma_db_path))
        if collection_name:
            self.collection = self.chroma_client.get_collection(collection_name)
        else:
            collections = self.chroma_client.list_collections()
            if not collections:
                raise RuntimeError(f"No Chroma collection found at {chroma_db_path}")
            self.collection = self.chroma_client.get_collection(collections[0].name)
        self.vector_store = ChromaVectorStore(chroma_collection=self.collection)

    def _contextual_from_meta(self, text: str, meta: Dict[str, Any]) -> str:
        company_name = str(meta.get("company_name", "?"))
        ticker = str(meta.get("ticker", "?"))
        form_type = str(meta.get("form_type", "?"))
        filing_date = str(meta.get("filing_date", "?"))[:10]
        filing_year = str(meta.get("filing_year", "?"))
        section_title = str(meta.get("section_title", "?"))
        return (
            f"Company: {company_name} ({ticker})\n"
            f"Filing: {form_type} | Date: {filing_date} | Year: {filing_year}\n"
            f"Section: {section_title}\n"
            f"Content: {text}"
        )

    def _bm25_mask(
        self,
        ticker: Optional[str] = None,
        filing_year: Optional[int] = None,
        form_type: Optional[str] = None,
    ) -> Optional[np.ndarray]:
        if not ticker and not filing_year and not form_type:
            return None
        mask = np.ones(len(self.df), dtype=float)
        if ticker:
            mask *= (self.df["ticker"].str.upper() == ticker.upper()).astype(float).values
        if filing_year:
            mask *= (self.df["filing_year"] == int(filing_year)).astype(float).values
        if form_type:
            mask *= (self.df["form_type"].str.upper() == form_type.upper()).astype(float).values
        if mask.sum() < 5:
            return None
        return mask

    def bm25_search(self, query: str, top_k: int, mask: Optional[np.ndarray] = None) -> List[RetrievedChunk]:
        scores = np.array(self.bm25.get_scores(query.lower().split()))
        if mask is not None:
            scores = scores * mask
        top_idx = np.argsort(scores)[::-1][:top_k]
        out: List[RetrievedChunk] = []
        for i in top_idx:
            if scores[i] <= 0:
                continue
            row = self.df.iloc[i]
            out.append(
                RetrievedChunk(
                    doc_name=row["doc_name"],
                    company=row["company"],
                    page_num=int(row["page_num"]),
                    chunk_id=row["chunk_id_str"],
                    raw_chunk=row["raw_chunk"],
                    contextual_chunk=row["contextual_chunk"],
                    score=float(scores[i]),
                    source="bm25",
                    ticker=row["ticker"],
                )
            )
        return out

    def dense_search(
        self,
        query: str,
        top_k: int,
        embed_model: SentenceTransformer,
        ticker: Optional[str] = None,
        filing_year: Optional[int] = None,
        form_type: Optional[str] = None,
    ) -> List[RetrievedChunk]:
        q_emb = embed_model.encode([query], normalize_embeddings=True)[0].tolist()

        filters = []
        if ticker:
            filters.append({"ticker": {"$eq": ticker.upper()}})
        if filing_year:
            filters.append({"filing_year": {"$eq": int(filing_year)}})
        if form_type:
            filters.append({"form_type": {"$eq": form_type.upper()}})

        kwargs: Dict[str, Any] = {
            "query_embeddings": [q_emb],
            "n_results": top_k,
            "include": ["documents", "metadatas", "distances"],
        }
        if len(filters) == 1:
            kwargs["where"] = filters[0]
        elif len(filters) > 1:
            kwargs["where"] = {"$and": filters}

        results = self.collection.query(**kwargs)
        docs = results.get("documents", [[]])[0]
        metas = results.get("metadatas", [[]])[0]
        dists = results.get("distances", [[]])[0]
        ids = results.get("ids", [[]])[0]

        out: List[RetrievedChunk] = []
        for doc_id, text, meta, dist in zip(ids, docs, metas, dists):
            meta_dict = dict(meta)
            doc_ticker = str(meta_dict.get("ticker", "?"))
            doc_form_type = str(meta_dict.get("form_type", "?"))
            filing_date = str(meta_dict.get("filing_date", "?"))[:10]
            out.append(
                RetrievedChunk(
                    doc_name=f"{doc_ticker}_{doc_form_type}_{filing_date}",
                    company=str(meta_dict.get("company_name", "?")),
                    page_num=safe_int(meta_dict.get("chunk_index", 0)),
                    chunk_id=str(doc_id),
                    raw_chunk=str(text),
                    contextual_chunk=self._contextual_from_meta(str(text), meta_dict),
                    score=1.0 - float(dist),
                    source="dense_chroma",
                    ticker=doc_ticker,
                )
            )
        return out

    def hybrid_search(
        self,
        query: str,
        embed_model: SentenceTransformer,
        reranker: CrossEncoder,
        bm25_top_k: int,
        dense_top_k: int,
        rerank_top_k: int,
        ticker: Optional[str] = None,
        filing_year: Optional[int] = None,
        form_type: Optional[str] = None,
    ) -> List[RetrievedChunk]:
        mask = self._bm25_mask(ticker=ticker, filing_year=filing_year, form_type=form_type)
        bm25_results = self.bm25_search(query, bm25_top_k, mask)
        dense_results = self.dense_search(query, dense_top_k, embed_model, ticker, filing_year, form_type)

        rrf: Dict[str, float] = {}
        pool: Dict[str, RetrievedChunk] = {}

        for rank, c in enumerate(bm25_results):
            rrf[c.chunk_id] = rrf.get(c.chunk_id, 0.0) + _rrf_score(rank)
            pool[c.chunk_id] = c
        for rank, c in enumerate(dense_results):
            rrf[c.chunk_id] = rrf.get(c.chunk_id, 0.0) + _rrf_score(rank)
            if c.chunk_id not in pool:
                pool[c.chunk_id] = c

        candidates = [pool[k] for k in sorted(rrf, key=rrf.__getitem__, reverse=True)]
        if not candidates:
            return []

        pairs = [(query, c.contextual_chunk) for c in candidates]
        scores = reranker.predict(pairs, show_progress_bar=False)
        for c, s in zip(candidates, scores):
            c.score = float(s)
            c.source = "hybrid_reranked"

        return sorted(candidates, key=lambda x: x.score, reverse=True)[:rerank_top_k]


def extract_retrieved_doc_names(chunks: List[RetrievedChunk]) -> List[str]:
    return list(dict.fromkeys([c.doc_name for c in chunks]))


def cleanup_planner_query(
    query: Optional[str],
    ticker: Optional[str] = None,
    filing_year: Optional[int] = None,
    form_type: Optional[str] = None,
) -> str:
    cleaned = (query or "").strip()
    if not cleaned:
        cleaned = "financial metric from SEC filing"
    upper = cleaned.upper()
    if upper.startswith("SELECT ") and " FROM " in upper:
        cleaned = cleaned.replace("SELECT", "").replace("FROM filings", "").strip()
    for token in ("WHERE", "FROM", "AND", "=", "'", '"'):
        cleaned = cleaned.replace(token, " ")
    cleaned = cleaned.replace("filing year", "filing_year")
    cleaned = cleaned.replace("|", " ")
    cleaned = cleaned.replace("_", " ")
    cleaned = cleaned.replace("company", "")
    cleaned = cleaned.replace("ticker", "")
    cleaned = " ".join(cleaned.split())

    parts = [cleaned]
    if ticker and ticker not in cleaned:
        parts.insert(0, ticker)
    if filing_year and str(filing_year) not in cleaned:
        parts.append(str(filing_year))
    if form_type and form_type not in cleaned:
        parts.append(form_type)
    return " ".join([p for p in parts if p]).strip()


def fails_retrieval_sanity_check(question: str, sub_queries: List[Dict[str, Any]], retrieved: List[RetrievedChunk]) -> bool:
    if not question.strip():
        return False
    if not CONFIG["enable_retrieval_sanity_check"]:
        return False
    if not retrieved:
        return False

    expected_tickers = {sq.get("ticker") for sq in sub_queries if sq.get("ticker")}
    if not expected_tickers:
        return False

    present_tickers = {c.ticker for c in retrieved if c.ticker}
    if not present_tickers:
        return False

    return not expected_tickers.issubset(present_tickers)


sec_chunks_path = resolve_path_from_env(CONFIG["sec_chunks_path"])
if not sec_chunks_path.exists():
    sec_chunks_path = (PROJECT_ROOT / "sec_rag_team_share/sec_data/chunks/sec_chunks.jsonl").resolve()

chroma_db_path = resolve_path_from_env(CONFIG["chroma_db_path"])
if not chroma_db_path.exists():
    chroma_db_path = (PROJECT_ROOT / "sec_rag_team_share/chroma_db").resolve()

sec_df = load_sec_chunks(sec_chunks_path)
chunk_dicts = sec_df_to_chunk_dicts(sec_df)

embed_model = SentenceTransformer(CONFIG["dense_model_name"])
reranker = CrossEncoder(CONFIG["reranker_model_name"])

global_index = CorpusIndex(
    chunks=chunk_dicts,
    chroma_db_path=chroma_db_path,
    collection_name=CONFIG["chroma_collection_name"],
)

print(f"Loaded SEC chunks: {len(sec_df):,}")
print(f"Chroma docs: {global_index.collection.count():,}")

Loaded SEC chunks: 12,606
Chroma docs: 12,606


In [3]:
class SubQuery(BaseModel):
    query: Optional[str] = Field(default=None, description="Search-optimized sub-query text for retrieval")
    ticker: Optional[str] = Field(default=None, description="Company ticker if company-specific")
    filing_year: Optional[int] = Field(default=None, description="Calendar year when the filing was submitted")
    form_type: Optional[str] = Field(default=None, description="10-K or 10-Q if specified")

    @field_validator("query", mode="before")
    @classmethod
    def normalize_query(cls, v):
        if isinstance(v, dict):
            parts: List[str] = []
            for key in ("query", "query_field", "metric", "line_item", "table", "topic"):
                value = v.get(key)
                if value is not None and str(value).strip():
                    parts.append(str(value).strip())
            fields = v.get("fields")
            if isinstance(fields, list) and fields:
                parts.append(" ".join(str(item).strip() for item in fields if str(item).strip()))
            return " | ".join(dict.fromkeys(parts)) if parts else None
        return v

    @field_validator("ticker", mode="before")
    @classmethod
    def normalize_ticker(cls, v):
        if v is None:
            return None
        cleaned = str(v).strip().upper()
        return cleaned or None

    @field_validator("form_type", mode="before")
    @classmethod
    def normalize_form_type(cls, v):
        if v is None:
            return None
        cleaned = str(v).strip().upper()
        return cleaned if cleaned in {"10-K", "10-Q"} else None


class PlannerOutput(BaseModel):
    needs_decomposition: bool = Field(
        description="True when the question requires multiple distinct filings, periods, or companies."
    )
    sub_queries: List[SubQuery] = Field(
        description="One sub-query for single-filing questions, otherwise 2-3 sub-queries split by company or period."
    )

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "sub_queries" not in normalized:
            for alias in ("queries", "sub_query", "query"):
                if alias in normalized:
                    value = normalized[alias]
                    normalized["sub_queries"] = value if isinstance(value, list) else [value]
                    break
        if "needs_decomposition" not in normalized:
            sub_queries = normalized.get("sub_queries", [])
            normalized["needs_decomposition"] = isinstance(sub_queries, list) and len(sub_queries) > 1
        return normalized

    @field_validator("needs_decomposition", mode="before")
    @classmethod
    def coerce_bool(cls, v):
        if isinstance(v, str):
            return v.strip().lower() in {"true", "1", "yes"}
        return bool(v)


class ContextEvalOutput(BaseModel):
    relevant: bool
    reason: str


class CriticOutput(BaseModel):
    decision: Literal["accept", "repair", "insufficient_data"]
    reason: str


class RepairOutput(BaseModel):
    decision: Literal["accept", "warn", "refuse"]
    revised_answer: str
    needs_new_retrieval: bool
    reason: str


class JudgeOutput(BaseModel):
    score: float
    claims_covered: float
    reason: str


COMPANY_TO_TICKER: Dict[str, str] = {
    "apple": "AAPL",
    "bank of america": "BAC",
    "berkshire hathaway": "BRK",
    "berkshire": "BRK",
    "costco": "COST",
    "chevron": "CVX",
    "johnson & johnson": "JNJ",
    "johnson and johnson": "JNJ",
    "jpmorgan chase": "JPM",
    "jpmorgan": "JPM",
    "microsoft": "MSFT",
    "nvidia": "NVDA",
    "unitedhealth": "UNH",
    "unitedhealth group": "UNH",
    "walmart": "WMT",
    "exxonmobil": "XOM",
    "exxon mobil": "XOM",
}


def infer_metadata_hints(text: str) -> Dict[str, Any]:
    lowered = (text or "").lower()
    ticker = None
    for name, mapped_ticker in COMPANY_TO_TICKER.items():
        if name in lowered:
            ticker = mapped_ticker
            break
    if not ticker:
        ticker_match = re.search(r"\b(AAPL|BAC|BRK|BRK-B|COST|CVX|JNJ|JPM|MSFT|NVDA|UNH|WMT|XOM)\b", text or "", flags=re.IGNORECASE)
        if ticker_match:
            ticker = ticker_match.group(1).upper().replace("-B", "")

    years = [int(match) for match in re.findall(r"\b(20\d{2})\b", text or "")]
    filing_year = years[0] if len(years) == 1 else None

    form_match = re.search(r"\b10-K\b|\b10-Q\b", text or "", flags=re.IGNORECASE)
    form_type = form_match.group(0).upper() if form_match else None

    return {
        "ticker": ticker,
        "filing_year": filing_year,
        "form_type": form_type,
    }


PLANNER_SYSTEM = (
    "You are a financial research planner working with SEC filings from these companies:\n"
    "AAPL (Apple), BAC (Bank of America), BRK (Berkshire Hathaway), COST (Costco), "
    "CVX (Chevron), JNJ (Johnson & Johnson), JPM (JPMorgan Chase), MSFT (Microsoft), "
    "NVDA (NVIDIA), UNH (UnitedHealth), WMT (Walmart), XOM (ExxonMobil).\n\n"
    "Decide whether the question requires data from multiple distinct filings, then produce sub-queries.\n\n"
    "Decompose (needs_decomposition=true) when the question explicitly compares different fiscal years or different companies.\n"
    "Do not decompose single-period, single-company questions.\n\n"
    "For each sub-query: set ticker if company-specific, filing_year if year-specific, and form_type when the question mentions 10-K or 10-Q.\n"
    "Every sub-query must include a non-empty query field. filing_year is the calendar year the filing was submitted.\n"
    "Return valid JSON only with keys needs_decomposition and sub_queries."
)

CONTEXT_EVAL_SYSTEM = (
    "You judge whether retrieved context is relevant and sufficient to answer a financial question. "
    "Mark as not relevant only if the context is clearly from the wrong company or period, or completely empty. "
    "Partial tables and incomplete passages still count as relevant if they contain the right metric. "
    "Return valid JSON only with keys relevant and reason."
)

GENERATOR_SYSTEM = (
    "You are a financial analyst answering questions using only the retrieved filing context. "
    "Be precise with numbers, units, fiscal periods, and line-item names. "
    "If the context does not contain the answer, say so explicitly rather than estimating."
)

CRITIC_SYSTEM = (
    "You are a critic for a financial RAG system. Evaluate the answer and choose one decision: "
    "accept, repair, or insufficient_data. Use repair when the data is present but the answer used it incorrectly. "
    "Use insufficient_data only when the required financial data is absent from the retrieved context. "
    "Return valid JSON only with keys decision and reason."
)

REPAIR_SYSTEM = (
    "You repair a financial RAG answer after critique. Default to accept with a revised answer whenever possible. "
    "Pay close attention to fiscal year, quarter, units, sign, and line-item name. "
    "Set needs_new_retrieval=true only if critical data is completely absent from context. "
    "Return valid JSON only with keys decision, revised_answer, needs_new_retrieval, and reason."
)


def _strip_code_fence(text: str) -> str:
    t = (text or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z0-9_\-]*", "", t).strip()
        if t.endswith("```"):
            t = t[:-3].strip()
    return t


def llm_complete(llm, prompt: str) -> str:
    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            rate_limiter.wait()
            return llm.complete(prompt).text.strip()
        except Exception:
            if attempt == CONFIG["llm_max_retries"] - 1:
                raise
            time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    raise RuntimeError("Unreachable")


def invoke_structured(llm, schema, system: str, user: str, fallback):
    prompt = (
        f"System:\n{system}\n\n"
        "Return ONLY valid JSON, with no markdown fences.\n\n"
        f"User:\n{user}"
    )
    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            raw = llm_complete(llm, prompt)
            data = json.loads(_strip_code_fence(raw))
            return schema.model_validate(data)
        except Exception:
            if attempt == CONFIG["llm_max_retries"] - 1:
                return fallback
            time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    return fallback


def invoke_text(llm, system: str, user: str, fallback_text: str = "") -> str:
    prompt = f"System:\n{system}\n\nUser:\n{user}"
    try:
        return llm_complete(llm, prompt)
    except Exception:
        return fallback_text


def format_retrieved_context(chunks: List[RetrievedChunk], max_chunks: int, max_chars: int) -> str:
    selected = chunks[:max_chunks]
    parts: List[str] = []
    for i, c in enumerate(selected, start=1):
        parts.append(
            f"[Doc {i}] Company: {c.company} | Filing: {c.doc_name}\n"
            f"Content: {c.raw_chunk}"
        )
    joined = "\n\n".join(parts)
    return joined[:max_chars]

In [4]:
class AgentStateRequired(TypedDict):
    question: str
    index: CorpusIndex


class AgentState(AgentStateRequired, total=False):
    rewritten_query: str
    sub_queries: List[Dict[str, Any]]
    needs_decomposition: bool

    retrieved_chunks: List[RetrievedChunk]
    retrieved_doc_names: List[str]
    retrieval_sanity_failed: bool

    context_retries: int
    context_eval_relevant: bool

    generated_answer: str

    critic_decision: str
    critic_reason: str

    repair_used: bool
    repair_decision: str
    repair_reason: str
    needs_new_retrieval: bool
    repair_retrieval_count: int
    is_repair_retrieval: bool

    final_answer: str


class PlanEvent(Event):
    state: AgentState


class RetrieveEvent(Event):
    state: AgentState


class RetrievedEvent(Event):
    state: AgentState


class GenerateEvent(Event):
    state: AgentState


class CritiqueEvent(Event):
    state: AgentState


class RepairEvent(Event):
    state: AgentState


class RepairRetrieveEvent(Event):
    state: AgentState


class SecAgenticRAGWorkflow(Workflow):
    @step
    async def query_planning(self, _ctx: Context, ev: StartEvent) -> PlanEvent:
        question = ev.get("question")
        index = ev.get("index")

        if not question or index is None:
            raise ValueError("Workflow requires question and index")

        state: AgentState = {
            "question": question,
            "index": index,
            "context_retries": 0,
            "repair_used": False,
            "repair_retrieval_count": 0,
            "is_repair_retrieval": False,
            "retrieval_sanity_failed": False,
            "sub_queries": [],
            "needs_decomposition": False,
        }

        result = invoke_structured(
            planner_llm,
            PlannerOutput,
            PLANNER_SYSTEM,
            f"Question:\n{question}",
            PlannerOutput(needs_decomposition=False, sub_queries=[SubQuery(query=question)]),
        )

        sub_queries: List[Dict[str, Any]] = []
        for sq in result.sub_queries:
            sq_dict = sq.model_dump()
            hints = infer_metadata_hints(question)
            query_text = sq_dict.get("query") or question
            query_hints = infer_metadata_hints(query_text)
            sq_dict["ticker"] = sq_dict.get("ticker") or query_hints.get("ticker") or hints.get("ticker")
            sq_dict["filing_year"] = sq_dict.get("filing_year") or query_hints.get("filing_year")
            sq_dict["form_type"] = sq_dict.get("form_type") or query_hints.get("form_type") or hints.get("form_type")
            if not sq_dict.get("query"):
                parts = [question]
                if sq_dict.get("ticker"):
                    parts.append(f"ticker {sq_dict['ticker']}")
                if sq_dict.get("filing_year"):
                    parts.append(f"filing year {sq_dict['filing_year']}")
                if sq_dict.get("form_type"):
                    parts.append(f"form {sq_dict['form_type']}")
                sq_dict["query"] = " | ".join(parts)
            sq_dict["query"] = cleanup_planner_query(
                sq_dict.get("query", ""),
                ticker=sq_dict.get("ticker"),
                filing_year=sq_dict.get("filing_year"),
                form_type=sq_dict.get("form_type"),
            )
            sub_queries.append(sq_dict)

        if not sub_queries:
            fallback_hints = infer_metadata_hints(question)
            sub_queries = [{
                "query": cleanup_planner_query(
                    question,
                    ticker=fallback_hints.get("ticker"),
                    filing_year=fallback_hints.get("filing_year"),
                    form_type=fallback_hints.get("form_type"),
                ),
                "ticker": fallback_hints.get("ticker"),
                "filing_year": fallback_hints.get("filing_year"),
                "form_type": fallback_hints.get("form_type"),
            }]

        state["sub_queries"] = sub_queries
        state["rewritten_query"] = sub_queries[0]["query"]
        state["needs_decomposition"] = result.needs_decomposition or len(sub_queries) > 1

        print(
            f"  [Planner] {'decomposed' if state['needs_decomposition'] else 'single'} | "
            f"{len(sub_queries)} sub-quer{'ies' if len(sub_queries) > 1 else 'y'}"
        )
        if state["needs_decomposition"]:
            for sq in sub_queries:
                print(
                    f"    -> {sq['query']}  "
                    f"(ticker={sq.get('ticker')}, year={sq.get('filing_year')}, form={sq.get('form_type')})"
                )

        # Event mapping: PlanEvent == LangGraph transition query_planner -> hybrid_retriever.
        return PlanEvent(state=state)

    @step
    async def hybrid_retriever(
        self,
        _ctx: Context,
        ev: PlanEvent | RetrieveEvent | RepairRetrieveEvent,
    ) -> RetrievedEvent:
        state = ev.state
        sub_queries = state.get("sub_queries", [])

        if isinstance(ev, RepairRetrieveEvent):
            state["is_repair_retrieval"] = True

        if len(sub_queries) <= 1:
            q = state.get("rewritten_query", state["question"])
            sq = sub_queries[0] if sub_queries else {}
            retrieved = state["index"].hybrid_search(
                query=q,
                embed_model=embed_model,
                reranker=reranker,
                bm25_top_k=CONFIG["bm25_top_k"],
                dense_top_k=CONFIG["dense_top_k"],
                rerank_top_k=CONFIG["rerank_top_k"],
                ticker=sq.get("ticker"),
                filing_year=sq.get("filing_year"),
                form_type=sq.get("form_type"),
            )
        else:
            per_k = CONFIG["decomposition_top_k_per_subquery"]
            merged: Dict[str, RetrievedChunk] = {}
            for sq in sub_queries:
                chunks = state["index"].hybrid_search(
                    query=sq["query"],
                    embed_model=embed_model,
                    reranker=reranker,
                    bm25_top_k=CONFIG["bm25_top_k"],
                    dense_top_k=CONFIG["dense_top_k"],
                    rerank_top_k=per_k,
                    ticker=sq.get("ticker"),
                    filing_year=sq.get("filing_year"),
                    form_type=sq.get("form_type"),
                )
                for chunk in chunks:
                    if chunk.chunk_id not in merged or chunk.score > merged[chunk.chunk_id].score:
                        merged[chunk.chunk_id] = chunk
            retrieved = sorted(merged.values(), key=lambda x: x.score, reverse=True)[: CONFIG["rerank_top_k"]]

        state["retrieved_chunks"] = retrieved
        state["retrieved_doc_names"] = extract_retrieved_doc_names(retrieved)
        state["retrieval_sanity_failed"] = fails_retrieval_sanity_check(
            state["question"],
            sub_queries,
            retrieved,
        )

        # Event mapping: RetrievedEvent == LangGraph post-retrieval state before context_evaluator.
        return RetrievedEvent(state=state)

    @step
    async def context_evaluator(self, _ctx: Context, ev: RetrievedEvent) -> GenerateEvent | RetrieveEvent:
        state = ev.state

        if state.get("retrieval_sanity_failed", False):
            relevant = False
        else:
            context = format_retrieved_context(
                state.get("retrieved_chunks", []),
                max_chunks=CONFIG["control_max_context_chunks"],
                max_chars=CONFIG["control_max_context_chars"],
            )
            judged = invoke_structured(
                context_eval_llm,
                ContextEvalOutput,
                CONTEXT_EVAL_SYSTEM,
                f"Question:\n{state['question']}\n\nContext:\n{context}",
                ContextEvalOutput(relevant=True, reason="fallback"),
            )
            relevant = judged.relevant

        state["context_eval_relevant"] = relevant

        if relevant:
            # Event mapping: GenerateEvent == LangGraph route_context("generator").
            return GenerateEvent(state=state)

        retries = state.get("context_retries", 0)
        if retries < CONFIG["max_context_retries"]:
            state["context_retries"] = retries + 1
            # Event mapping: RetrieveEvent == LangGraph route_context("retry_retrieval") -> increment_context_retry -> hybrid_retriever.
            return RetrieveEvent(state=state)

        # Same behavior as LangGraph: force forward after retry cap.
        return GenerateEvent(state=state)

    @step
    async def generator(self, _ctx: Context, ev: GenerateEvent) -> CritiqueEvent:
        state = ev.state
        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["generator_max_context_chunks"],
            max_chars=CONFIG["generator_max_context_chars"],
        )
        answer = invoke_text(
            generator_llm,
            GENERATOR_SYSTEM,
            f"Question:\n{state['question']}\n\nRetrieved context:\n{context}",
            fallback_text="",
        ).strip()

        state["generated_answer"] = answer
        state["final_answer"] = answer

        # Event mapping: CritiqueEvent == LangGraph generator -> critic transition.
        return CritiqueEvent(state=state)

    @step
    async def critic(self, _ctx: Context, ev: CritiqueEvent) -> StopEvent | RepairEvent:
        state = ev.state

        if state.get("is_repair_retrieval", False):
            # Event mapping: StopEvent == LangGraph route_critic("end") after repair retrieval pass.
            return StopEvent(result=state)

        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["control_max_context_chunks"],
            max_chars=CONFIG["control_max_context_chars"],
        )

        result = invoke_structured(
            critic_llm,
            CriticOutput,
            CRITIC_SYSTEM,
            (
                f"Question:\n{state['question']}\n\n"
                f"Context:\n{context}\n\n"
                f"Answer:\n{state.get('generated_answer', '')}"
            ),
            CriticOutput(decision="accept", reason="fallback"),
        )

        state["critic_decision"] = result.decision
        state["critic_reason"] = result.reason

        if result.decision == "insufficient_data":
            state["final_answer"] = (
                "Insufficient data: The required information could not be found in the "
                f"retrieved SEC filings. ({result.reason})"
            )
            # Event mapping: StopEvent == LangGraph route_critic("end") for insufficient_data.
            return StopEvent(result=state)

        if result.decision == "repair":
            # Event mapping: RepairEvent == LangGraph route_critic("repair").
            return RepairEvent(state=state)

        # accept -> end
        return StopEvent(result=state)

    @step
    async def repair(self, _ctx: Context, ev: RepairEvent) -> StopEvent | RepairRetrieveEvent:
        state = ev.state
        context = format_retrieved_context(
            state.get("retrieved_chunks", []),
            max_chunks=CONFIG["control_max_context_chunks"],
            max_chars=CONFIG["control_max_context_chars"],
        )

        result = invoke_structured(
            repair_llm,
            RepairOutput,
            REPAIR_SYSTEM,
            (
                f"Question:\n{state['question']}\n\n"
                f"Context:\n{context}\n\n"
                f"Answer:\n{state.get('generated_answer', '')}\n\n"
                f"Critique:\n{state.get('critic_reason', 'Needs repair')}"
            ),
            RepairOutput(
                decision="accept",
                revised_answer=state.get("generated_answer", ""),
                needs_new_retrieval=False,
                reason="fallback",
            ),
        )

        state["repair_used"] = True
        state["repair_decision"] = result.decision
        state["repair_reason"] = result.reason
        state["needs_new_retrieval"] = result.needs_new_retrieval
        state["final_answer"] = (result.revised_answer or "").strip()

        if result.needs_new_retrieval:
            count = state.get("repair_retrieval_count", 0) + 1
            state["repair_retrieval_count"] = count
            if count <= CONFIG["max_repair_retrievals"]:
                state["is_repair_retrieval"] = True
                # Event mapping: RepairRetrieveEvent == LangGraph repair -> mark_repair_retrieval -> hybrid_retriever.
                return RepairRetrieveEvent(state=state)

        # Event mapping: StopEvent == LangGraph route_repair("end").
        return StopEvent(result=state)

In [5]:
async def run_agentic_rag(question: str, index: CorpusIndex) -> Dict[str, Any]:
    wf = SecAgenticRAGWorkflow(timeout=600, verbose=True)
    result = await wf.run(question=question, index=index)
    return dict(result)


# Quick demo
# question = "How did Apple's gross margin percentage change between fiscal year 2023 and fiscal year 2024?"
# result = await run_agentic_rag(question, global_index)
#
# print("Question:", question)
# print("Final answer:\n", result.get("final_answer", ""))
# print("Critic decision:", result.get("critic_decision"))
# print("Repair used:", result.get("repair_used"), "| Repair retrieval count:", result.get("repair_retrieval_count"))

In [6]:
question = "How did Apple's gross margin percentage change between fiscal year 2023 and fiscal year 2024?"
result = await run_agentic_rag(question, global_index)

print("Question:", question)
print("Final answer:\n", result.get("final_answer", ""))
print("Critic decision:", result.get("critic_decision"))
print("Repair used:", result.get("repair_used"), "| Repair retrieval count:", result.get("repair_retrieval_count"))
print("Retrieved docs:", result.get("retrieved_doc_names", []))

Running step query_planning
  [Planner] decomposed | 2 sub-queries
    -> How did Apple s gross margin percentage change between fiscal year 2023 and fiscal year 2024? AAPL filing year 2023 form 10-K  (ticker=AAPL, year=2023, form=10-K)
    -> How did Apple s gross margin percentage change between fiscal year 2023 and fiscal year 2024? AAPL filing year 2024 form 10-K  (ticker=AAPL, year=2024, form=10-K)
Step query_planning produced event <class '__main__.PlanEvent'>
Running step hybrid_retriever
Step hybrid_retriever produced event <class '__main__.RetrievedEvent'>
Running step context_evaluator
Step context_evaluator produced event <class '__main__.GenerateEvent'>
Running step generator
Step generator produced event <class '__main__.CritiqueEvent'>
Running step critic
Step critic produced event <class 'workflows.events.StopEvent'>
Question: How did Apple's gross margin percentage change between fiscal year 2023 and fiscal year 2024?
Final answer:
 Insufficient data: The required infor